# AI_Agent_Workshop_Day2_01 — Problem Setup and Data

This notebook introduces the municipal challenge for Day 2:

> Build an AI-powered assistant that helps residents understand which level of government is responsible for a service and what to do next.

We will start with a compact, curated dataset and a reproducible schema before building the agent.

## Learning goals

By the end of this notebook, students should be able to:

1. Frame the service-routing problem as a structured AI task.
2. Define an output schema for a public-service assistant.
3. Inspect and normalize a starter service catalog.
4. Export clean artifacts for later retrieval and evaluation.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = Path("day2_workshop")

DATA_DIR = ROOT / "data"
EVAL_DIR = ROOT / "eval"
ARTIFACTS_DIR = ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

print("Root:", ROOT.resolve())
print("Data files:", [p.name for p in DATA_DIR.iterdir()])

## The challenge as a machine learning task

A resident question becomes an input example.

We want the assistant to return:

- the most likely service name,
- the responsible level of government,
- the likely responsible body,
- a concise explanation,
- and the next steps.

This is a hybrid task involving semantic matching, classification, retrieval, and answer generation.

In [ ]:
response_schema = {
    "service_name": "string",
    "jurisdiction_level": "City | Region | Province | Federal | Mixed | Unclear",
    "responsible_body": "string",
    "confidence": "float in [0, 1]",
    "reasoning_summary": "short grounded explanation",
    "next_steps": ["step 1", "step 2"],
    "sources": ["url 1", "url 2"]
}
response_schema

## Load the starter catalog

In [ ]:
catalog = pd.read_csv(DATA_DIR / "service_catalog.csv")
catalog

In [ ]:
catalog.info()

In [ ]:
catalog.sample(min(len(catalog), 5), random_state=42)

## Normalize the service catalog

We will add a few cleaned fields that make retrieval easier later.

In [ ]:
def normalize_catalog(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["service_name_normalized"] = out["service_name"].str.lower().str.strip()
    out["keywords_list"] = out["keywords"].fillna("").apply(
        lambda x: [k.strip().lower() for k in x.split(";") if k.strip()]
    )
    out["retrieval_text"] = (
        out["service_name"].fillna("") + " | " +
        out["description"].fillna("") + " | " +
        out["keywords"].fillna("")
    ).str.lower()
    return out

clean_catalog = normalize_catalog(catalog)
clean_catalog.head()

## Inspect jurisdictional coverage

In [ ]:
clean_catalog[["service_name", "jurisdiction_level", "responsible_body"]].sort_values(
    by=["jurisdiction_level", "service_name"]
)

## Export cleaned artifacts

In [ ]:
output_path = ARTIFACTS_DIR / "service_catalog.cleaned.json"
clean_catalog.to_json(output_path, orient="records", indent=2)
print("Saved:", output_path.resolve())

## Evaluation set preview

In [ ]:
eval_df = pd.read_csv(EVAL_DIR / "service_eval_set.csv")
eval_df

## Exercise

1. Add 3 more services to the catalog.
2. Create at least 2 ambiguous examples.
3. Mark one example as `Unclear` or `Mixed`.
4. Re-export the cleaned catalog.

## Reflection

- Which parts of this problem are classification problems?
- Which parts require retrieval?
- Which parts benefit from tool use?
- Which outputs should be deterministic and machine-readable?